<a href="https://colab.research.google.com/github/ItsUgesh/wine-type-svm/blob/main/wine_type_svm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

def load_wine_csv(name):
    url = f"https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-{name}.csv"
    try:
        return pd.read_csv(url, sep=";")
    except Exception:
        return pd.read_csv(f"winequality-{name}.csv", sep=";")

red = load_wine_csv("red")
white = load_wine_csv("white")
red["type"] = "red"
white["type"] = "white"
data = pd.concat([red, white], ignore_index=True)

print(data.head())
print(data.columns.tolist())
data.info()
print(data.describe().round(2))
print("Missing values:", data.isnull().sum().sum())

   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.4              0.70         0.00             1.9      0.076   
1            7.8              0.88         0.00             2.6      0.098   
2            7.8              0.76         0.04             2.3      0.092   
3           11.2              0.28         0.56             1.9      0.075   
4            7.4              0.70         0.00             1.9      0.076   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 11.0                  34.0   0.9978  3.51       0.56   
1                 25.0                  67.0   0.9968  3.20       0.68   
2                 15.0                  54.0   0.9970  3.26       0.65   
3                 17.0                  60.0   0.9980  3.16       0.58   
4                 11.0                  34.0   0.9978  3.51       0.56   

   alcohol  quality type  
0      9.4        5  red  
1      9.8        5  red  
2    

In [2]:
y = data["type"]
X = data.drop(columns="type")
print(y.value_counts())
print("Features:", X.shape[1])

type
white    4898
red      1599
Name: count, dtype: int64
Features: 12


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on training data only
X_test_scaled = scaler.transform(X_test)
print("Train:", X_train_scaled.shape, "| Test:", X_test_scaled.shape)

Train: (5197, 12) | Test: (1300, 12)


In [4]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

models = {
    "linear": SVC(kernel="linear"),
    "rbf": SVC(kernel="rbf"),
    "poly (degree 3)": SVC(kernel="poly", degree=3),
}

for name, m in models.items():
    m.fit(X_train_scaled, y_train)
    pred = m.predict(X_test_scaled)
    print(f"{name:16s} accuracy = {accuracy_score(y_test, pred):.4f}")

linear           accuracy = 0.9977
rbf              accuracy = 0.9969
poly (degree 3)  accuracy = 0.9946


In [5]:
rows = []
for kernel in ["linear", "rbf", "poly"]:
    for C in [0.5, 1, 5]:
        m = SVC(kernel=kernel, C=C, degree=3).fit(X_train_scaled, y_train)
        acc = accuracy_score(y_test, m.predict(X_test_scaled))
        rows.append({"kernel": kernel, "C": C, "accuracy": round(acc, 4)})

pd.DataFrame(rows).pivot(index="kernel", columns="C", values="accuracy")

C,0.5,1.0,5.0
kernel,,,
linear,0.9977,0.9977,0.9977
poly,0.9915,0.9946,0.9954
rbf,0.9962,0.9969,0.9969


## 1. Which kernel performed best?
The Linear kernel performed best, with an accuracy of 99.77% (3 wrong out of
1,300 test wines). RBF was almost identical at 99.69% (4 wrong), and Polynomial
was slightly lower at 99.46% (7 wrong). The gap between Linear and RBF is a
single wine, so the two kernels are effectively tied; only the Polynomial kernel
is clearly behind (0.31 percentage points, about 4 more wines).

## 2. Why might one kernel be better than the others?
- **Linear** separates the classes with a flat boundary. It performing as well as
  the more flexible kernels shows that red and white wines are almost linearly
  separable: a straight plane in the standardized feature space is enough.
- **RBF** can curve the boundary around groups of points, but here there was
  almost nothing left to gain from the extra flexibility.
- **Polynomial** draws a curved boundary with a fixed shape. It was the least
  accurate, probably because its curve is less well matched to this data and it
  is more sensitive to its settings.
- Red and white wines differ strongly in their chemistry (for example, white
  wines usually have much more sulfur dioxide), so all three kernels score above
  99%. For comparison, always guessing "white" would give about 75%.

## 3. What happens when C changes to 0.5 or 5?
C controls how strictly the model punishes mistakes on the training data.
- **Linear:** no change (99.77% for every C, 3 wrong). The data is so cleanly
  separated that C does not matter.
- **RBF:** a tiny gain from C = 0.5 to C = 1 (99.62% to 99.69%, one wine), then
  no further change.
- **Polynomial:** the clearest effect. Accuracy rises with C (99.15%, 99.46%,
  99.54%; 11, 7 and 6 wrong wines). A small C makes the polynomial boundary too
  smooth and it underfits, while a larger C lets it fit the training data more
  closely.
Overall the models are not very sensitive to C, except the polynomial kernel.

## Conclusion
The Linear kernel is the best choice: it has the highest accuracy (99.77%) for
every C, it is the simplest and fastest model, and it does not need tuning. RBF
is a close second. Because the test set has only 1,300 wines, differences of one
or two wines (about 0.1 points) are within noise and should not be over-interpreted.